# Getting Started with Interactive Analytics

This notebook benchmarks Snowflake Interactive Tables and Interactive Warehouses against Standard Warehouses, comparing query latency and throughput across sequential and concurrent workloads.

## Set common variables

Set database name, interactive warehouse name and standard warehouse name that will be used throughout the notebook

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
USER = session.sql("SELECT CURRENT_USER()").collect()[0][0]
DB_NAME = f'{USER}_MY_DEMO_DB'
INTERACTIVE_WH_NAME = f'{USER}_INT_WH'
STANDARD_WH_NAME = f'{USER}_STD_WH'

print(f"User: {USER}\nDatabase: {DB_NAME}\nInteractive WH: {INTERACTIVE_WH_NAME}\nStandard WH: {STANDARD_WH_NAME}")

## Set up role, warehouse, and database

Interactive Warehouses and Interactive Tables are now generally available (GA) and enabled by default on your account, so there's no need to check the Snowflake version or verify any account parameters.

Using a SQL cell, we'll set the active role and create the standard warehouse (`STANDARD_WH_NAME`), database (`DB_NAME`), and schemas used throughout this notebook. All statements use `IF NOT EXISTS`, so this cell is safe to re-run.

> **Note:** In a Snowflake Notebook, SQL and Python cells share the same session, so any `USE ROLE`, `USE DATABASE`, or `USE WAREHOUSE` statement you run in a SQL cell also applies to subsequent Python cells (and vice versa).

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

-- Create the compute and database objects used throughout this notebook (idempotent)
CREATE WAREHOUSE IF NOT EXISTS {{STANDARD_WH_NAME}} WITH WAREHOUSE_SIZE = 'X-SMALL';
CREATE DATABASE IF NOT EXISTS {{DB_NAME}};

CREATE SCHEMA IF NOT EXISTS {{DB_NAME}}.BENCHMARK_FDN;
CREATE SCHEMA IF NOT EXISTS {{DB_NAME}}.BENCHMARK_INTERACTIVE;

USE WAREHOUSE {{STANDARD_WH_NAME}};
USE DATABASE {{DB_NAME}};

## Create an interactive warehouse

Next, let's create our interactive warehouse using a SQL cell:

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE INTERACTIVE WAREHOUSE {{INTERACTIVE_WH_NAME}}
    WAREHOUSE_SIZE = 'XSMALL'
    MIN_CLUSTER_COUNT = 1
    MAX_CLUSTER_COUNT = 1
    COMMENT = 'Interactive warehouse demo';

## Data Setup & Loading

The next cell creates the `HITS2_CSV` table and loads it from the `synthetic_hits_data.csv` file bundled with this notebook. The load is **idempotent**: it checks whether the table already contains data and, if so, skips the load on subsequent runs.

> **Note:** The data is loaded from the bundled CSV using `pandas` + `write_pandas` (no external network access required). Make sure `synthetic_hits_data.csv` is added to this notebook's files.

In [ ]:
%%sql -r dataframe_9
USE WAREHOUSE {{STANDARD_WH_NAME}};

In [ ]:
import pandas as pd

DB, SCHEMA, TABLE = DB_NAME, "BENCHMARK_FDN", "HITS2_CSV"
FQ = f"{DB}.{SCHEMA}.{TABLE}"
CSV_FILE = "synthetic_hits_data.csv"  # bundled with this notebook

# Create the source table if it doesn't already exist
session.sql(f"""
CREATE TABLE IF NOT EXISTS {FQ} (
    EventDate DATE,
    CounterID INT,
    ClientIP STRING,
    SearchEngineID INT,
    SearchPhrase STRING,
    ResolutionWidth INT,
    Title STRING,
    IsRefresh INT,
    DontCountHits INT
)
""").collect()

# Idempotent load: only load when the table is empty
row_count = session.sql(f"SELECT COUNT(*) FROM {FQ}").collect()[0][0]
if row_count > 0:
    print(f"{FQ} already has {row_count:,} rows. Skipping data load.")
else:
    print(f"Loading data into {FQ} ...")
    pdf = pd.read_csv(CSV_FILE)
    pdf["EventDate"] = pd.to_datetime(pdf["EventDate"]).dt.date    
    session.write_pandas(pdf, TABLE, database=DB, schema=SCHEMA, quote_identifiers=False)
    row_count = session.sql(f"SELECT COUNT(*) FROM {FQ}").collect()[0][0]
    print(f"Loaded {row_count:,} rows into {FQ}.")

In [ ]:
%%sql -r dataframe_3
USE WAREHOUSE {{STANDARD_WH_NAME}};
SELECT * FROM {{DB_NAME}}.BENCHMARK_FDN.HITS2_CSV;

## Create an interactive table

Now, we'll use the standard warehouse to efficiently create our new interactive `CUSTOMERS` table by copying all the data from the original standard table:

In [ ]:
%%sql -r dataframe_4
-- Use a standard warehouse to build the interactive table's data
USE WAREHOUSE {{STANDARD_WH_NAME}};
CREATE SCHEMA IF NOT EXISTS {{DB_NAME}}.BENCHMARK_INTERACTIVE;

CREATE OR REPLACE INTERACTIVE TABLE
  {{DB_NAME}}.BENCHMARK_INTERACTIVE.CUSTOMERS CLUSTER BY (ClientIP)
AS
  SELECT * FROM {{DB_NAME}}.BENCHMARK_FDN.HITS2_CSV;

## Attach interactive table to a warehouse

Next, we'll attach our interactive table to the warehouse, which pre-warms the data cache for optimal query performance:

> **Note:** `ADD TABLES` is a performance optimization, not a requirement. It proactively warms the warehouse's data cache so queries avoid a cold start. Any table you don't attach is still queryable and gets cached on demand the first time it's accessed. Proactive warming is currently limited to 10 tables.

In [ ]:
%%sql -r dataframe_5
USE DATABASE {{DB_NAME}};
ALTER WAREHOUSE {{INTERACTIVE_WH_NAME}} ADD TABLES(BENCHMARK_INTERACTIVE.CUSTOMERS);

## Configure a fallback warehouse

Interactive warehouses are tuned for short, sub-second queries, so Snowflake fixes their statement timeout at a maximum of 5 seconds and automatically cancels any query that runs longer. To make sure an occasional heavy or ad-hoc query still completes instead of failing, you can designate a **fallback warehouse**: a standard warehouse that automatically re-runs any query that exceeds the 5-second timeout on the interactive warehouse.

This retry is transparent to the client (it behaves as an internal retry), so the query still returns its result. It keeps fast dashboard queries responsive while isolating them from the occasional long-running query.

We'll reuse the standard warehouse created earlier as the fallback, then confirm the setting via the `FALLBACK_WAREHOUSE` column.

> **Note:** The fallback is a **standard** warehouse (size it the same as or larger than the interactive warehouse) and must be started or set to auto-resume to accept retried queries. The querying role needs `USAGE` on both warehouses. To remove it later, run `ALTER WAREHOUSE {{INTERACTIVE_WH_NAME}} UNSET FALLBACK_WAREHOUSE;`.

In [ ]:
%%sql -r dataframe_6
ALTER WAREHOUSE {{INTERACTIVE_WH_NAME}} SET FALLBACK_WAREHOUSE = {{STANDARD_WH_NAME}};

SHOW WAREHOUSES LIKE '{{INTERACTIVE_WH_NAME}}';

## Run queries with interactive warehouse

Now, we'll run our first performance test on the interactive setup by executing a page-view query, timing its execution, and then plotting the results.

We'll start by activating the interactive warehouse and disabling the result cache using a SQL cell:

In [ ]:
%%sql -r dataframe_7
USE WAREHOUSE {{INTERACTIVE_WH_NAME}};
USE DATABASE {{DB_NAME}};
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

Before proceeding further, we'll create a helper function for data visualization.

In [ ]:
import matplotlib.pyplot as plt

def plot_data(data, title, time_taken, color='#29B5E8'):
    # Separate titles and counts
    titles = [item[0] for item in data]
    counts = [item[1] for item in data]

    # Plot bar chart
    
    plt.figure(figsize=(12, 4))
    plt.bar(titles, counts, color=color)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel("Counts")
    plt.xlabel("Title")
    plt.title(title)
    plt.text(0.5, 1.5, f'Time taken: {time_taken:.4f} seconds',
         ha='center', va='top',
         transform=plt.gca().transAxes,
         fontdict={'size': 16})
    #plt.tight_layout()
    plt.show()

Next, in a Python cell we'll run a query to find the top 10 most viewed pages for July 2013, measure how long it takes, and then plot the results and execution time:

In [ ]:
import time

cursor = session.connection.cursor()

query = """
SELECT Title, COUNT(*) AS PageViews
FROM BENCHMARK_INTERACTIVE.CUSTOMERS
WHERE CounterID = 62
  AND EventDate >= '2013-07-01'
  AND EventDate <= '2013-07-31'
  AND DontCountHits = 0
  AND IsRefresh = 0
  AND Title <> ''
  AND REGEXP_LIKE(Title, '^[\\x00-\\x7F]+$')
  AND LENGTH(Title) < 20
GROUP BY Title
ORDER BY PageViews DESC
LIMIT 10;
"""

start_time = time.time()
result = cursor.execute(query).fetchall()
end_time = time.time()
time_taken = end_time - start_time

plot_data(result, "Page visit analysis (Interactive)", time_taken)

## Compare to a standard warehouse

To establish a performance baseline, we'll run an identical page-view query on a standard warehouse to measure and plot its results for comparison.

We'll start by preparing the session for a performance benchmark by selecting the standard warehouse (`STANDARD_WH_NAME`), disabling the result cache, and setting the active database using a SQL cell:

In [ ]:
%%sql -r dataframe_8
USE WAREHOUSE {{STANDARD_WH_NAME}};
USE DATABASE {{DB_NAME}};
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

Here, in a Python cell we'll run a top 10 page views analysis by executing the query, measuring its performance, and immediately plotting the results and execution time:

In [ ]:
query = """
SELECT Title, COUNT(*) AS PageViews
FROM BENCHMARK_FDN.HITS2_CSV
WHERE CounterID = 62
  AND EventDate >= '2013-07-01'
  AND EventDate <= '2013-07-31'
  AND DontCountHits = 0
  AND IsRefresh = 0
  AND Title <> ''
  AND REGEXP_LIKE(Title, '^[\\x00-\\x7F]+$')
  AND LENGTH(Title) < 20
GROUP BY Title
ORDER BY PageViews DESC
LIMIT 10;
"""

start_time = time.time()
result = cursor.execute(query).fetchall()
end_time = time.time()
time_taken = end_time - start_time

plot_data(result, "Page visit analysis (Standard)", time_taken, '#5B5B5B')


## Sequential Query Benchmark

To directly compare performance, we'll benchmark both the interactive and standard warehouses over 50 sequential runs and then plot their latencies side-by-side in a grouped bar chart.

In [ ]:
import numpy as np

runs = 50

def run_and_measure(count, mode):
    wh = INTERACTIVE_WH_NAME if mode == "iw" else STANDARD_WH_NAME
    table = "BENCHMARK_INTERACTIVE.CUSTOMERS" if mode == "iw" else "BENCHMARK_FDN.HITS2_CSV"
    query = f"""
        SELECT SearchEngineID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth)
        FROM {table}
        WHERE SearchPhrase <> ''
        GROUP BY SearchEngineID, ClientIP
        ORDER BY c DESC LIMIT 10
    """
    cursor.execute(f"USE WAREHOUSE {wh}")
    cursor.execute('ALTER SESSION SET USE_CACHED_RESULT = FALSE;')

    timings = []
    for _ in range(count + 1):
        t0 = time.time()
        cursor.execute(query).fetchall()
        timings.append(time.time() - t0)
    return timings[1:] # skip warm-up run

counts_iw = run_and_measure(runs, "iw")
print(counts_iw)

counts_std = run_and_measure(runs, "std")
print(counts_std)

In [ ]:
import matplotlib.pyplot as plt
titles = [(i+1) for i in range(0, len(counts_iw))]

x = np.arange(len(titles))  # the label locations
width = 0.35  # bar width

fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(x - width/2, counts_std, width, label="Standard", color="#5B5B5B")
ax.bar(x + width/2, counts_iw, width, label="Interactive", color="#29B5E8")

ax.set_ylabel("Latency")
ax.set_xlabel("Query run")
ax.set_title("Standard vs Interactive warehouse")
ax.set_xticks(x)
ax.set_xticklabels(titles)
ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=2
)
plt.show()

In [ ]:
# Calculate means and standard deviations for error bars
mean_std = np.mean(counts_std)
mean_iw = np.mean(counts_iw)
std_std = np.std(counts_std)
std_iw = np.std(counts_iw)

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(["Standard", "Interactive"], [mean_std, mean_iw],
              yerr=[std_std, std_iw], capsize=8,
              color=["#5B5B5B", "#29B5E8"], width=0.5)

ax.set_ylabel("Latency (seconds)")
ax.set_title("Standard vs Interactive warehouse\n(mean over {} runs with std dev)".format(len(counts_std)))
plt.tight_layout()
plt.show()

---

## Concurrent Query Benchmark

Simulates real-world concurrent query load by:
- Using a **mixed query pool** (light, medium, heavy)
- **Staggered arrival** via Poisson distribution
- **Ramping up** concurrency from 1 to 8 workers
- Measuring **server-side latency** (p50/p90/p99) and **throughput** (queries/sec)

In [ ]:
import random, time, numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

QUERY_TEMPLATES = {
    "light": "SELECT * FROM {table} WHERE CounterID = 62 LIMIT 1",
    "medium": """SELECT SearchEngineID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth)
        FROM {table} WHERE SearchPhrase <> ''
        GROUP BY SearchEngineID, ClientIP ORDER BY c DESC LIMIT 10""",
    "heavy": """SELECT EventDate, COUNT(*) AS hits, COUNT(DISTINCT ClientIP) AS unique_ips,
        AVG(ResolutionWidth), SUM(CASE WHEN SearchPhrase <> '' THEN 1 ELSE 0 END)
        FROM {table} GROUP BY EventDate ORDER BY EventDate""",
}

def build_query_pool(table):
    return [(k, v.format(table=table)) for k, v in QUERY_TEMPLATES.items()]

def worker(conn, wh_name, query_pool, n_queries=6, arrival_rate=5):
    cur = conn.cursor()
    cur.execute(f"USE WAREHOUSE {wh_name}")
    cur.execute("ALTER SESSION SET USE_CACHED_RESULT = FALSE")
    latencies = []
    for _ in range(n_queries):
        time.sleep(random.expovariate(arrival_rate))
        _, query = random.choice(query_pool)
        cur.execute(query).fetchall()
        qid = cur.sfqid
        ms = cur.execute(f"SELECT TOTAL_ELAPSED_TIME FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION()) WHERE QUERY_ID = '{qid}'").fetchone()[0]
        latencies.append(ms / 1000.0)
    cur.close()
    return latencies

def run_concurrent_benchmark(conn, wh_name, table, concurrency_levels, rounds=10):
    query_pool = build_query_pool(table)
    all_results = []
    for n in concurrency_levels:
        print(f"  concurrency={n} ({rounds} rounds) ...", end=" ", flush=True)
        round_stats = []
        for _ in range(rounds):
            t0 = time.time()
            with ThreadPoolExecutor(max_workers=n) as pool:
                futures = [pool.submit(worker, conn, wh_name, query_pool) for _ in range(n)]
                lats = [l for f in as_completed(futures) for l in f.result()]
            wall = time.time() - t0
            round_stats.append({"p50": np.percentile(lats, 50), "p90": np.percentile(lats, 90),
                                "p99": np.percentile(lats, 99), "throughput_qps": len(lats) / wall})
        result = {"concurrency": n}
        for m in ["p50", "p90", "p99", "throughput_qps"]:
            vals = [r[m] for r in round_stats]
            result[f"{m}_mean"], result[f"{m}_std"] = np.mean(vals), np.std(vals)
        all_results.append(result)
        print(f"p50={result['p50_mean']:.3f}s(±{result['p50_std']:.3f})  p90={result['p90_mean']:.3f}s(±{result['p90_std']:.3f})  qps={result['throughput_qps_mean']:.1f}(±{result['throughput_qps_std']:.1f})")
    return all_results

concurrency_levels = [1, 2, 4, 8]
conn = session.connection

print("Interactive warehouse:")
results_iw = run_concurrent_benchmark(conn, INTERACTIVE_WH_NAME, "BENCHMARK_INTERACTIVE.CUSTOMERS", concurrency_levels)

print("\nStandard warehouse:")
results_std = run_concurrent_benchmark(conn, STANDARD_WH_NAME, "BENCHMARK_FDN.HITS2_CSV", concurrency_levels)

print(f"\nBenchmark complete ({run_concurrent_benchmark.__defaults__[0]} rounds per level).")

### Visualizing the Results

Now that we have latency and throughput data across multiple rounds for each concurrency level, we plot two side-by-side charts:

- **Left — Concurrency vs Latency:** Shows how p50, p90, and p99 latency change as we increase the number of concurrent workers. Error bars represent standard deviation across rounds, indicating how stable each measurement is. Ideally, lines stay flat — meaning the warehouse handles more load without slowing down.
- **Right — Concurrency vs Throughput:** Shows how many queries per second (QPS) each warehouse sustains at each concurrency level. Taller bars are better. We expect throughput to grow linearly with workers — a plateau would indicate the warehouse is saturated.

In [ ]:
import matplotlib.pyplot as plt

benchmark_rounds = run_concurrent_benchmark.__defaults__[0]
levels = [r["concurrency"] for r in results_iw]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for results, label, color in [(results_std, "Standard", "#5B5B5B"), (results_iw, "Interactive", "#29B5E8")]:
    for metric, marker, ls, alpha in [("p50", "o", "-", 1.0), ("p90", "s", "--", 0.6), ("p99", "^", ":", 0.4)]:
        ax1.errorbar(levels, [r[f"{metric}_mean"] for r in results],
                     yerr=[r[f"{metric}_std"] for r in results],
                     fmt=f"{marker}{ls}", color=color, alpha=alpha, capsize=4, label=f"{label} {metric}")

ax1.set(xlabel="Concurrent Workers", ylabel="Latency (seconds)", xticks=levels)
ax1.set_ylim(bottom=0)
ax1.set_title(f"Concurrency vs Latency (lower is better)\nmean ± std over {benchmark_rounds} rounds")
ax1.legend(fontsize=7, ncol=2)
ax1.grid(True, alpha=0.3)

x = np.arange(len(levels))
w = 0.35
for results, label, color, offset in [(results_std, "Standard", "#5B5B5B", -w/2), (results_iw, "Interactive", "#29B5E8", w/2)]:
    ax2.bar(x + offset, [r["throughput_qps_mean"] for r in results], w,
            yerr=[r["throughput_qps_std"] for r in results], capsize=4, label=label, color=color)

ax2.set(xlabel="Concurrent Workers", ylabel="Queries / Second", xticks=x)
ax2.set_xticklabels(levels)
ax2.set_title(f"Concurrency vs Throughput (higher is better)\nmean ± std over {benchmark_rounds} rounds")
ax2.legend()
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

### Interpreting the Benchmark

The following cell dynamically analyzes the benchmark data and generates an interpretation. It compares Interactive vs Standard across every concurrency level, detects scaling issues, tail latency spikes, and throughput plateaus. When Interactive underperforms, it provides actionable suggestions for improvement. The interpretation updates automatically whenever the benchmark is re-run with different parameters.

In [ ]:
from IPython.display import display, Markdown

benchmark_rounds = run_concurrent_benchmark.__defaults__[0]
total_levels = len(results_iw)
max_level = results_iw[-1]["concurrency"]

# Per-level comparisons
p50_advs = [(1 - iw["p50_mean"] / std["p50_mean"]) * 100 for iw, std in zip(results_iw, results_std)]
avg_adv = np.mean(p50_advs)
latency_wins = sum(1 for a in p50_advs if a > 0)
throughput_wins = sum(1 for iw, std in zip(results_iw, results_std) if iw["throughput_qps_mean"] > std["throughput_qps_mean"])

# Scaling
iw_p50s = [r["p50_mean"] for r in results_iw]
std_p50s = [r["p50_mean"] for r in results_std]
iw_spread, std_spread = (max(p) - min(p) for p in [iw_p50s, std_p50s])
iw_scales_well, std_scales_well = iw_spread < 0.05, std_spread < 0.05

# Tail latency
iw_p99_max = max(r["p99_mean"] for r in results_iw)
std_p99_max = max(r["p99_mean"] for r in results_std)
iw_tail_spike = iw_p99_max > 3 * min(iw_p50s)
std_tail_spike = std_p99_max > 3 * min(std_p50s)

# Throughput
iw_peak = results_iw[-1]["throughput_qps_mean"]
std_peak = results_std[-1]["throughput_qps_mean"]
qps_diff = (iw_peak / std_peak - 1) * 100
iw_qps = [r["throughput_qps_mean"] for r in results_iw]
std_qps = [r["throughput_qps_mean"] for r in results_std]
iw_plateau = len(iw_qps) > 2 and iw_qps[-1] < iw_qps[-2] * 1.3
std_plateau = len(std_qps) > 2 and std_qps[-1] < std_qps[-2] * 1.3

# Reliability
iw_cv = np.mean([r["p50_std"] / r["p50_mean"] for r in results_iw]) * 100
std_cv = np.mean([r["p50_std"] / r["p50_mean"] for r in results_std]) * 100

# --- Build output ---
lines = ["### Interpretation (dynamically generated from benchmark data)\n"]

lines.append("**Latency:**\n")
winner_label = "Interactive" if avg_adv > 0 else "Standard"
wins = latency_wins if avg_adv > 0 else total_levels - latency_wins
lines.append(f"- {winner_label} wins on p50 latency in **{wins}/{total_levels}** concurrency levels")
for iw, std, adv in zip(results_iw, results_std, p50_advs):
    c = iw["concurrency"]
    w = "Interactive" if adv > 0 else "Standard"
    lines.append(f"- At {c} worker{'s' if c > 1 else ''}: **{abs(adv):.0f}% lower** for {w} "
                 f"(Interactive: {iw['p50_mean']*1000:.0f}ms vs Standard: {std['p50_mean']*1000:.0f}ms)")
lines.append(f"- Average p50 advantage: **{abs(avg_adv):.0f}%** for {winner_label}")

lines.append("\n**Scaling:**\n")
for name, ok, p50s in [("Interactive", iw_scales_well, iw_p50s), ("Standard", std_scales_well, std_p50s)]:
    lo, hi = min(p50s)*1000, max(p50s)*1000
    lines.append(f"- {name} p50 **{'stays flat' if ok else 'increases noticeably'}** from 1→{max_level} workers "
                 f"(min: {lo:.0f}ms, max: {hi:.0f}ms, spread: {hi-lo:.0f}ms)")

if iw_tail_spike or std_tail_spike:
    lines.append("\n**Tail Latency:**\n")
    for name, spike, p99, p50_min in [("Interactive", iw_tail_spike, iw_p99_max, min(iw_p50s)),
                                       ("Standard", std_tail_spike, std_p99_max, min(std_p50s))]:
        if spike:
            lines.append(f"- {name} shows p99 spikes of **{p99*1000:.0f}ms** (>{3*p50_min*1000:.0f}ms = 3× best p50)")
        else:
            lines.append(f"- {name} tail latency is well-controlled (p99: {p99*1000:.0f}ms)")

lines.append("\n**Throughput:**\n")
lines.append(f"- Interactive wins in **{throughput_wins}/{total_levels}** concurrency levels")
qps_label = "roughly equal" if abs(qps_diff) < 5 else f"**{abs(qps_diff):.0f}% higher** for {'Interactive' if qps_diff > 0 else 'Standard'}"
lines.append(f"- At {max_level} workers: Interactive **{iw_peak:.1f} qps** vs Standard **{std_peak:.1f} qps** — {qps_label}")
for name, p in [("Interactive", iw_plateau), ("Standard", std_plateau)]:
    if p: lines.append(f"- {name} shows a throughput plateau — approaching its concurrency ceiling")

lines.append("\n**Reliability:**\n")
lines.append(f"- Based on **{benchmark_rounds} rounds** per concurrency level")
lines.append(f"- Coefficient of variation (p50): Interactive **{iw_cv:.1f}%**, Standard **{std_cv:.1f}%**")
cv_max = max(iw_cv, std_cv)
lines.append(f"- {'Tight error bars — high confidence' if cv_max < 15 else 'Moderate variance — directionally reliable' if cv_max < 30 else 'High variance — consider more rounds'}")

lines.append("\n**Takeaway:**\n")
if latency_wins > total_levels / 2 and avg_adv > 20:
    lines.append("- Interactive warehouses are **clearly favorable** — lower latency and better throughput under concurrent mixed workloads")
elif latency_wins >= total_levels / 2:
    lines.append("- Interactive warehouses show a **moderate advantage** in this benchmark")
else:
    lines.append("- Standard warehouses **outperform Interactive** in this benchmark")

suggestions = []
if avg_adv < 20:
    suggestions.append(f"**Increase warehouse size** — X-Small limits cache (avg p50: {np.mean(iw_p50s)*1000:.0f}ms)")
if iw_tail_spike:
    suggestions.append(f"**Reduce heavy query weight** — cache overwhelmed (p99: {iw_p99_max*1000:.0f}ms)")
if not iw_scales_well:
    suggestions.append(f"**Enable multi-cluster scaling** — spread of {iw_spread*1000:.0f}ms suggests contention")
if iw_plateau:
    suggestions.append("**Increase `MAX_CONCURRENCY_LEVEL`** — default 8, can go up to 64")
if avg_adv < 0:
    suggestions.append("**Verify table is cached** — use `ALTER WAREHOUSE ... ADD TABLES(...)`")
    suggestions.append("**Check data size vs cache** — partition or filter to a hot subset")
if suggestions:
    lines.append("\n**Suggestions to improve Interactive performance:**\n")
    lines.extend(f"- {s}" for s in suggestions)

display(Markdown("\n".join(lines)))

### Metrics Glossary

- **p50 (Median Latency)** — The middle value when all query latencies are sorted. 50% of queries finish faster, 50% slower. Best single measure of "typical" query speed.

  `p50 = sorted(latencies)[len(latencies) // 2]`

- **p90 (90th Percentile Latency)** — 90% of queries finish within this time. Captures the experience of most users, including those hitting slower queries.

  `p90 = sorted(latencies)[int(len(latencies) * 0.90)]`

- **p99 (99th Percentile Latency)** — 99% of queries finish within this time. Reveals worst-case outliers — the "tail latency" that affects 1 in 100 users.

  `p99 = sorted(latencies)[int(len(latencies) * 0.99)]`

- **QPS (Queries Per Second)** — How many queries the warehouse completes per second of wall-clock time. Measures throughput capacity.

  `qps = total_queries_completed / total_wall_time_seconds`

- **Coefficient of Variation (CV)** — Standard deviation divided by mean, expressed as a percentage. Measures how consistent/reproducible the results are across rounds. Lower = more stable.

  `cv = (std_dev / mean) × 100%`

- **Spread** — The difference between the highest and lowest p50 across concurrency levels. Measures how much latency changes as load increases. A small spread means the warehouse scales well under pressure.

  `spread = max(p50 across levels) − min(p50 across levels)`